In [1]:
import pandas as pd
import ast
import os

# ── STEP 1: INITIALIZE ───────────────────────────────────────────────────────
movies_df = pd.DataFrame()
IN_FILE = "tmdb_cleaned_movies.csv"
OUT_FILE = "tmdb_featured_movies.csv"

def get_primary_genre(gs):
    try:
        genres_list = ast.literal_eval(gs)
        return genres_list[0] if isinstance(genres_list, list) and genres_list else 'Unknown'
    except Exception:
        return 'Unknown'

# ── STEP 2: PIPELINE ─────────────────────────────────────────────────────────
if os.path.exists(IN_FILE):
    movies_df = pd.read_csv(IN_FILE)
    print(f"📦 Loaded {len(movies_df):,} records.")

    # Filtering for Commercial English-Language Cinema
    mask = (movies_df['budget'] >= 100_000) & (movies_df['revenue'] >= 100_000) & (movies_df['original_language'] == 'en')
    movies_df = movies_df[mask].copy()
    
    # Feature Extraction
    movies_df['release_date'] = pd.to_datetime(movies_df['release_date'])
    movies_df['release_month'] = movies_df['release_date'].dt.month
    movies_df['profit'] = movies_df['revenue'] - movies_df['budget']
    movies_df['roi_percentage'] = (movies_df['profit'] / movies_df['budget']) * 100
    movies_df['primary_genre'] = movies_df['genre_names'].apply(get_primary_genre)
    
    # Insight Summary
    print(f"✅ Success! Final Cleaned Shape: {movies_df.shape}")
    cols = ['title', 'release_month', 'primary_genre', 'budget', 'roi_percentage']
    print(movies_df.sort_values(by='revenue', ascending=False)[cols].head(5))
    
    movies_df.to_csv(OUT_FILE, index=False)
    print(f"\n💾 Saved output to: {OUT_FILE}")
else:
    print(f"❌ ERROR: Missing '{IN_FILE}'")

📦 Loaded 756,402 records.
✅ Success! Final Cleaned Shape: (334810, 19)
                               title  release_month    primary_genre  \
139945                        Avatar             12           Action   
480900             Avengers: Endgame              4        Adventure   
620407      Avatar: The Way of Water             12  Science Fiction   
79964                        Titanic             11            Drama   
375106  Star Wars: The Force Awakens             12        Adventure   

             budget  roi_percentage  
139945  237000000.0     1133.631235  
480900  356000000.0      686.359298  
620407  350000000.0      562.928652  
79964   200000000.0     1032.081177  
375106  245000000.0      744.172908  

💾 Saved output to: tmdb_featured_movies.csv
